In [33]:
"""
Memory-efficient NPI data loading options.
The full file is ~11 GB / 9.4M rows — loading all at once crashes the kernel.
Use one of the approaches below.
"""
import pandas as pd

CSV_PATH = "npidata_pfile_20050523-20260208.csv"

# Columns to load
COLS = [
    "NPI",
    "Provider Organization Name (Legal Business Name)",
    "Provider First Line Business Mailing Address",
    "Provider Business Mailing Address State Name",
    "Provider First Line Business Practice Location Address",
    "Provider Business Practice Location Address City Name",
    "Provider Business Practice Location Address State Name",
    "Provider Business Practice Location Address Postal Code",
    "Provider Business Mailing Address Postal Code",
    "Provider Business Practice Location Address Country Code (If outside U.S.)"
]

## Option 1: Load only essential columns (recommended)
Reduces memory by ~80–90% by loading a subset of columns.

In [41]:
COUNTRY_COL = "Provider Business Practice Location Address Country Code (If outside U.S.)"
dtype_map = {
    "NPI": "str",
    "Provider Business Practice Location Address Postal Code": "str",
    "Provider Business Mailing Address Postal Code": "str",
}

# Chunked read + filter for US only (avoids loading non-US into memory)
chunks = []
for chunk in pd.read_csv(CSV_PATH, usecols=COLS, dtype=dtype_map, chunksize=500_000):
    chunks.append(chunk[chunk[COUNTRY_COL] == "US"])
df = pd.concat(chunks, ignore_index=True)
# Drop PR (Puerto Rico) and VI (U.S. Virgin Islands)
state_col = "Provider Business Mailing Address State Name"
df = df[~df[state_col].isin(["PR", "VI"])].copy()
print(f"Loaded {len(df):,} US rows × {len(df.columns)} columns (excl. PR, VI)")
df.head()

Loaded 8,971,949 US rows × 10 columns (excl. PR, VI)


,NPI,Provider Organization Name (Legal Business Name),Provider First Line Business Mailing Address,Provider Business Mailing Address State Name,Provider Business Mailing Address Postal Code,Provider First Line Business Practice Location Address,Provider Business Practice Location Address City Name,Provider Business Practice Location Address State Name,Provider Business Practice Location Address Postal Code,Provider Business Practice Location Address Country Code (If outside U.S.)
0,1679576722,NaN,PO BOX 2168,NE,688482168,3500 CENTRAL AVE,KEARNEY,NE,688472944,US
1,1588667638,NaN,1824 KING STREET,FL,322044736,1824 KING STREET,JACKSONVILLE,FL,322044736,US
2,1497758544,"CUMBERLAND COUNTY HOSPITAL SYSTEM, INC",3418 VILLAGE DR,NC,283044552,3418 VILLAGE DR,FAYETTEVILLE,NC,283044552,US
3,1215930367,NaN,17323 RED OAK DR,TX,770901243,17323 RED OAK DR,HOUSTON,TX,770901243,US
4,1023011178,COLLABRIA CARE,414 S JEFFERSON ST,CA,945594515,414 S JEFFERSON ST,NAPA,CA,945594515,US


## Create short_ZIP from postal codes (5-digit US ZIP)
US-only data. Practice postal 3–9 digits → 5-digit short_ZIP:
- **3–4 digits:** left-pad to 5 (742→00742, 1742→01742)
- **5 digits:** use as is (10003→10003)
- **6 digits:** first 2, pad to 5
- **7 digits:** first 3, pad to 5 (6591323→00659)
- **8 digits:** first 4, pad to 5 (40110896→04011)
- **9 digits:** first 5 (170331402→17033)

In [48]:
import re

PRACTICE_POSTAL = "Provider Business Practice Location Address Postal Code"
MAILING_POSTAL = "Provider Business Mailing Address Postal Code"

def to_short_zip(val):
    """Convert practice postal to 5-digit short_ZIP per digit-length rules."""
    if pd.isna(val) or val == "":
        return ""
    if isinstance(val, (int, float)):
        val = int(val)
    s = re.sub(r"\D", "", str(val))
    if len(s) == 0:
        return ""
    n = len(s)
    if n <= 4:
        return s.zfill(5)
    if n == 5:
        return s
    if n == 6:
        return s[:2].zfill(5)
    if n == 7:
        return s[:3].zfill(5)
    if n == 8:
        return s[:4].zfill(5)
    return s[:5]  # n >= 9

# Digit-length helper (needed before drop)
def digit_len(val):
    if pd.isna(val) or val == "": return 0
    if isinstance(val, (int, float)): val = int(val)
    return len(re.sub(r"\D", "", str(val)))

digits = df[PRACTICE_POSTAL].apply(digit_len)
df = df[digits > 0].copy()
digits = df[PRACTICE_POSTAL].apply(digit_len)  # recompute after drop
print(f"Dropped {(digits == 0).sum():,} rows with 0-digit practice postal. Remainder: {len(df):,} rows")

# Force override for specific indices FIRST (before formula)
df["short_ZIP"] = ""
df.at[6507233, "short_ZIP"] = "10087"
df.at[4535337, "short_ZIP"] = "08625"

# Apply to_short_zip only for rows still empty
mask = df["short_ZIP"] == ""
df.loc[mask, "short_ZIP"] = df.loc[mask, PRACTICE_POSTAL].apply(to_short_zip)
mask = df["short_ZIP"] == ""
df.loc[mask, "short_ZIP"] = df.loc[mask, MAILING_POSTAL].apply(to_short_zip)

# Digit-length distribution
print("\nDigit-length distribution (practice postal):")
print(digits.value_counts().sort_index().to_string())

# 1-4 digits: left-padded to 5
mask_pad = (digits >= 1) & (digits <= 4)
n_pad = mask_pad.sum()
print(f"\nSanity: 1-4 digits (left-padded to 5) — {n_pad:,} rows found")
if n_pad > 0:
    display(df.loc[mask_pad, [PRACTICE_POSTAL, MAILING_POSTAL, "short_ZIP"]].head(10))
else:
    print("  (US addresses typically have 5 or 9-digit ZIPs)")

# 7-digit and 8-digit data
mask_7 = digits == 7
mask_8 = digits == 8
print(f"\nSanity: 7-digit (first 3, pad to 5) — {mask_7.sum():,} rows")
display(df.loc[mask_7, [PRACTICE_POSTAL, MAILING_POSTAL, "short_ZIP"]].head(10))
print(f"\nSanity: 8-digit (first 4, pad to 5) — {mask_8.sum():,} rows")
display(df.loc[mask_8, [PRACTICE_POSTAL, MAILING_POSTAL, "short_ZIP"]].head(10))

Dropped 0 rows with 0-digit practice postal. Remainder: 8,971,932 rows

Digit-length distribution (practice postal):
Provider Business Practice Location Address Postal Code
5     684176
7          2
8         13
9    8287741

Sanity: 1-4 digits (left-padded to 5) — 0 rows found
  (US addresses typically have 5 or 9-digit ZIPs)

Sanity: 7-digit (first 3, pad to 5) — 2 rows


,Provider Business Practice Location Address Postal Code,Provider Business Mailing Address Postal Code,short_ZIP
5659437,1334311,133430011,00133
7103471,8012434,806201011,00801



Sanity: 8-digit (first 4, pad to 5) — 13 rows


,Provider Business Practice Location Address Postal Code,Provider Business Mailing Address Postal Code,short_ZIP
1755645,11702323,115905114,01170
4535337,08625863,191445261,08625
4975839,60444000,600900457,06044
5093980,84646461,846460461,08464
5410302,67855637,678550637,06785
5466184,91762224,917305807,09176
5588146,45853417,458530417,04585
6263096,81620202,816310622,08162
6507233,10087-332,977021289,10087
6539579,30084678,303411780,03008


## Count retailers in Provider Organization Name
Case-insensitive counts for Walgreens/Walgreen and major pharmacy/grocery chains.

In [49]:
org_col = "Provider Organization Name (Legal Business Name)"

# Walgreens or Walgreen (same stem)
retailers = [
    ("Walgreens / Walgreen", "Walgreen"),  # matches both
    ("CVS", "CVS"),
    ("Safeway", "Safeway"),
    ("Pathmark", "Pathmark"),
    ("Kaiser Permanente", "Kaiser Permanente"),
    ("Kroger", "Kroger"),
    ("ShopRite", "ShopRite"),
    ("Costco", "Costco"),
    ("Health Mart", "Health Mart"),
    ("Good Neighbor", "Good Neighbor"),
    ("Walmart", "Walmart"),
    ("Rite Aid", "Rite Aid"),
]

counts = []
for name, pattern in retailers:
    cnt = df[org_col].str.contains(pattern, case=False, na=False).sum()
    counts.append({"Retailer": name, "Count": cnt})

summary = pd.DataFrame(counts)
summary.style.format({"Count": "{:,}"})

,Retailer,Count
0,Walgreens / Walgreen,"11,745"
1,CVS,"8,742"
2,Safeway,806
3,Pathmark,104
4,Kaiser Permanente,224
5,Kroger,"1,389"
6,ShopRite,99
7,Costco,"1,885"
8,Health Mart,46
9,Good Neighbor,50


## One-hot encoding for retailers
Columns: 1 if retailer found in org name, 0 otherwise. `non_chain` = 1 when none are found.

In [51]:
# (column_name, pattern) for one-hot encoding
RETAILER_COLS = [
    ("Walgreens", "Walgreen"),
    ("CVS", "CVS"),
    ("Safeway", "Safeway"),
    ("Pathmark", "Pathmark"),
    ("Kaiser_Permanente", "Kaiser Permanente"),
    ("Kroger", "Kroger"),
    ("ShopRite", "ShopRite"),
    ("Costco", "Costco"),
    ("Health_Mart", "Health Mart"),
    ("Good_Neighbor", "Good Neighbor"),
    ("Walmart", "Walmart"),
    ("Rite_Aid", "Rite Aid"),
]

# Append one-hot columns to original df (in-place)
for col_name, pattern in RETAILER_COLS:
    df[col_name] = df[org_col].str.contains(pattern, case=False, na=False).astype(int)

df["non_chain"] = (df[[c for c, _ in RETAILER_COLS]].sum(axis=1) == 0).astype(int)

# Filtered: Costco only (Costco column == 1)
df_costco = df[df["Costco"] == 1]
print(f"Costco rows: {len(df_costco):,}")
df_costco.head(15)

Costco rows: 1,885


,NPI,Provider Organization Name (Legal Business Name),Provider First Line Business Mailing Address,Provider Business Mailing Address State Name,Provider Business Mailing Address Postal Code,Provider First Line Business Practice Location Address,Provider Business Practice Location Address City Name,Provider Business Practice Location Address State Name,Provider Business Practice Location Address Postal Code,Provider Business Practice Location Address Country Code (If outside U.S.),...,Pathmark,Kaiser_Permanente,Kroger,ShopRite,Costco,Health_Mart,Good_Neighbor,Walmart,Rite_Aid,non_chain
965030,1619087210,COSTCO WHOLESALE CORPORATION,PO BOX 34300,WA,981241300,2207 W COMMONWEALTH AVE,ALHAMBRA,CA,918031302,US,...,0,0,0,0,1,0,0,0,0,0
965031,1497865000,COSTCO WHOLESALE CORPORATION,PO BOX 34300,WA,981241300,17900 NEWHOPE ST,FOUNTAIN VALLEY,CA,92708,US,...,0,0,0,0,1,0,0,0,0,0
965032,1912017526,COSTCO WHOLESALE CORPORATION,PO BOX 34300,WA,981241300,7900 W QUINCY AVE,LITTLETON,CO,80123,US,...,0,0,0,0,1,0,0,0,0,0
965033,1558471169,COSTCO WHOLESALE CORPORATION,PO BOX 34300,WA,981241300,200 FEDERAL RD,BROOKFIELD,CT,06804,US,...,0,0,0,0,1,0,0,0,0,0
965035,1528178134,COSTCO WHOLESALE CORPORATION,PO BOX 34300,WA,981241300,4901 GATE PKWY,JACKSONVILLE,FL,32246,US,...,0,0,0,0,1,0,0,0,0,0
965036,1982714598,COSTCO WHOLESALE CORPORATION,PO BOX 34300,WA,981241300,645 BARRETT PKWY,KENNESAW,GA,30144,US,...,0,0,0,0,1,0,0,0,0,0
965038,1316057920,COSTCO WHOLESALE CORPORATION,PO BOX 34300,WA,981241300,505 W ARMY TRAIL RD,BLOOMINGDALE,IL,60108,US,...,0,0,0,0,1,0,0,0,0,0
965039,1043320658,COSTCO WHOLESALE CORPORATION,PO BOX 34300,WA,981241300,250 N RANDALL RD,LAKE IN THE HILLS,IL,60156,US,...,0,0,0,0,1,0,0,0,0,0
965040,1942310560,COSTCO WHOLESALE CORPORATION,PO BOX 34300,WA,981241300,71 2ND AVE,WALTHAM,MA,02451,US,...,0,0,0,0,1,0,0,0,0,0
965041,1477663094,COSTCO WHOLESALE CORPORATION,PO BOX 34300,WA,981241300,12011 TECHNOLOGY DR,EDEN PRAIRIE,MN,55344,US,...,0,0,0,0,1,0,0,0,0,0
